# QM 640 Capstone — Step 5: Market Model, AR, CAR, CAAR

Implements the event-study mechanics from the Synopsis's Analytic Approach:

- Market model (estimated on day -150 to -31): `R_it = alpha_i + beta_i * R_mt + epsilon_it`
- Abnormal return (day 0 onward): `AR_it = R_it - (alpha_i + beta_i * R_mt)`
- Cumulative abnormal return: `CAR_i = sum(AR_it)` over the event window
  - `CAR_short`: day +1 to +3 (primary DV for RQ1-RQ4)
  - `CAR_long`: day +1 to +30 (robustness check only)
- CAAR (headline number for RQ1): `mean(CAR_i)` across all events

## Cell 1 — Clone (or pull) the repo

Run this first, every session. `BASE_DIR` is the single fixed path every
other cell in this notebook reads from and writes to.

In [1]:
import os
from google.colab import userdata

GITHUB_USERNAME = "Shanmuganathan75"      # edit if different
REPO_NAME = "QM640-WALSH-CAPSTONE"        # must match the repo URL EXACTLY (hyphens included)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # set once via Colab Secrets (key icon, left sidebar)

BASE_DIR = f"/content/{REPO_NAME}"
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    # Wipe any stale/incomplete folder from a previous failed attempt before retrying
    !rm -rf {BASE_DIR}
    !git clone {remote_url} {BASE_DIR}
else:
    !git -C {BASE_DIR} pull

# Fail loudly instead of silently continuing with a non-git folder -
# this is exactly the bug that caused "fatal: not a git repository" earlier.
if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    raise RuntimeError(
        f"Clone failed: no .git folder found at {BASE_DIR}.\n"
        f"Check that GITHUB_USERNAME (\'{GITHUB_USERNAME}\') and REPO_NAME "
        f"(\'{REPO_NAME}\') exactly match your repo URL (case and hyphens "
        f"included), and that GITHUB_TOKEN is set in Colab Secrets with "
        f"Contents: Read and write access."
    )

for sub in ["scripts", "data/raw", "data/processed/returns", "results"]:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

!git -C {BASE_DIR} config user.email "Shan_muganathan@yahoo.com"
!git -C {BASE_DIR} config user.name "Shanmuganathan Ekambaram"

print("Repo ready at:", BASE_DIR)

Cloning into '/content/QM640-WALSH-CAPSTONE'...
remote: Enumerating objects: 1107, done.
remote: Counting objects: 100% (294/294), done.
remote: Compressing objects: 100% (153/153), done.
remote: Total 1107 (delta 122), reused 232 (delta 89), pack-reused 813 (from 1)
Receiving objects: 100% (1107/1107), 8.15 MiB | 12.01 MiB/s, done.
Resolving deltas: 100% (563/563), done.
Repo ready at: /content/QM640-WALSH-CAPSTONE


## Cell 2 — Install dependencies

In [2]:
!pip install -q pandas numpy statsmodels

## Cell 3 — Configuration

In [3]:
import os

RETURNS_DIR = os.path.join(BASE_DIR, "data/processed/returns")
SCREENING_FILE = os.path.join(BASE_DIR, "data/raw/screening_worksheet.csv")
FIRM_SIZE_FILE = os.path.join(BASE_DIR, "data/raw/firm_size.csv")
SECTOR_REFERENCE_FILE = os.path.join(BASE_DIR, "data/raw/sector_reference.csv")
OUTPUT_FILE = os.path.join(BASE_DIR, "data/processed/analysis_dataset.csv")

ESTIMATION_WINDOW = (-150, -31)   # trading days relative to event
SHORT_WINDOW = (1, 3)
LONG_WINDOW = (1, 30)


## Cell 4 — Market model + CAR functions

In [4]:
import pandas as pd
import numpy as np
import statsmodels.api as sm


def estimate_market_model(df):
    """Fits R_it = alpha + beta * R_mt on the estimation window only."""
    est = df[
        (df["trading_day_offset"] >= ESTIMATION_WINDOW[0]) &
        (df["trading_day_offset"] <= ESTIMATION_WINDOW[1])
    ].dropna(subset=["daily_return_firm", "daily_return_market"])

    if len(est) < 60:
        return None, None

    X = sm.add_constant(est["daily_return_market"])
    y = est["daily_return_firm"]
    model = sm.OLS(y, X).fit()
    alpha, beta = model.params["const"], model.params["daily_return_market"]
    return alpha, beta


def compute_car(df, alpha, beta, window):
    """Sums abnormal returns over the given (start, end) trading-day window."""
    w = df[
        (df["trading_day_offset"] >= window[0]) &
        (df["trading_day_offset"] <= window[1])
    ].dropna(subset=["daily_return_firm", "daily_return_market"])

    if w.empty:
        return np.nan

    ar = w["daily_return_firm"] - (alpha + beta * w["daily_return_market"])
    return ar.sum()

## Cell 4b — Direct gap-fill for tickers missing firm size / sector

**Fix (Interim Report feedback — Dataset Completeness, was 15/509 = 2.9% matched):** the root cause was that `02_index_constituents.ipynb` only pulls market cap and sector for S&P 500 / Russell 3000 **member** tickers, then Step 5 joined by ticker - so any confirmed-event firm outside those two index lists silently got `None` for both fields. `yfinance` can return market cap and sector for **any** public ticker regardless of index membership, so this step identifies exactly which confirmed-event tickers are missing from the index-based reference tables and fetches those directly - a much smaller, targeted list (typically tens of tickers, not the full ~3,000-ticker index universe), so it's fast even inside this notebook.

Sector taxonomies differ across sources (GICS labels like "Information Technology"; Yahoo's own taxonomy uses "Technology"), so the tech/non-tech classification below normalizes across all three sources rather than relying on exact string matches.

In [5]:
import yfinance as yf
import time

TECH_SECTOR_LABELS = {
    "information technology", "communication services", "technology",
    "software", "internet content & information", "financial technology",
}

def classify_sector_binary(sector_value):
    """Normalizes across GICS, Russell/iShares, and Yahoo's own sector
    taxonomies (they use different label strings) into one Technology /
    Non-Technology split, instead of relying on a single taxonomy's exact
    label strings as the previous version did."""
    if sector_value is None or (isinstance(sector_value, float) and pd.isna(sector_value)):
        return None
    return "Technology" if str(sector_value).strip().lower() in TECH_SECTOR_LABELS else "Non-Technology"


def gap_fill_firm_size_and_sector(missing_tickers, headers=None, pause=0.5):
    """Direct per-ticker yfinance lookup for tickers the index-based
    reference tables (firm_size.csv, sector_reference.csv) don't cover.
    Mirrors the logic in 02_index_constituents.ipynb's get_market_cap, kept
    local here so Step 5 doesn't depend on re-running Step 2."""
    rows = []
    for i, t in enumerate(missing_tickers, start=1):
        cap, sector = None, None
        try:
            fi = yf.Ticker(t).fast_info
            cap = fi.get("market_cap") if hasattr(fi, "get") else getattr(fi, "market_cap", None)
        except Exception:
            cap = None
        try:
            info = yf.Ticker(t).info
            if not cap:
                cap = info.get("marketCap")
            sector = info.get("sector")
        except Exception as e:
            print(f"  gap-fill skip {t}: {e}")

        rows.append({
            "ticker": t,
            "market_cap_usd": cap,
            "market_cap_log": np.log(cap) if cap and cap > 0 else None,
            "sector": sector,
        })
        if i % 25 == 0:
            print(f"  gap-fill processed {i}/{len(missing_tickers)}")
        time.sleep(pause)
    return pd.DataFrame(rows)


## Cell 5 — Process all events into the final analysis dataset

In [6]:
import glob

files = glob.glob(os.path.join(RETURNS_DIR, "*.csv"))
print(f"Processing {len(files)} event return files ...")

def safe_read_csv(path, required_cols, label):
    """Loads a CSV, but degrades gracefully instead of crashing the whole
    pipeline if the file is missing or empty (e.g. an upstream step like
    Step 2's yfinance market-cap pull failed silently). Downstream code
    already handles missing values per-row via `.empty` checks, so an
    empty placeholder just means those columns come back as None."""
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        print(f"WARNING: {label} ({path}) is missing or empty - "
              f"proceeding with an empty placeholder. Re-run the notebook "
              f"that generates it before your real (non-smoke-test) run.")
        return pd.DataFrame(columns=required_cols)
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        print(f"WARNING: {label} ({path}) has no parseable columns - "
              f"proceeding with an empty placeholder.")
        return pd.DataFrame(columns=required_cols)


screening = pd.read_csv(SCREENING_FILE)
firm_size = safe_read_csv(FIRM_SIZE_FILE, ["ticker", "market_cap_usd", "market_cap_log"], "firm_size.csv")
sector_reference = safe_read_csv(SECTOR_REFERENCE_FILE, ["ticker", "sector", "sector_source"], "sector_reference.csv")

# --- Gap-fill pass: find tickers among THIS run's actual events that the
#     index-based reference tables don't cover, and fetch them directly ---
event_tickers = sorted({os.path.basename(f).replace(".csv", "").split("_", 1)[0] for f in files})
missing_size = [t for t in event_tickers if t not in set(firm_size["ticker"])]
missing_sector = [t for t in event_tickers if t not in set(sector_reference["ticker"])]
missing_any = sorted(set(missing_size) | set(missing_sector))

if missing_any:
    print(f"\n{len(missing_any)}/{len(event_tickers)} event tickers not covered by "
          f"the index-based reference tables - gap-filling directly via yfinance ...")
    gap_filled = gap_fill_firm_size_and_sector(missing_any)

    gap_size = gap_filled.dropna(subset=["market_cap_usd"])[["ticker", "market_cap_usd", "market_cap_log"]].copy()
    gap_size["firm_size_source"] = "Direct gap-fill (yfinance)"
    firm_size["firm_size_source"] = "Index reference (Step 2)"
    firm_size = pd.concat([firm_size, gap_size], ignore_index=True).drop_duplicates(subset=["ticker"], keep="first")

    gap_sector = gap_filled.dropna(subset=["sector"])[["ticker", "sector"]].copy()
    gap_sector["sector_source"] = "Direct gap-fill (yfinance)"
    sector_reference = pd.concat([sector_reference, gap_sector], ignore_index=True).drop_duplicates(subset=["ticker"], keep="first")

    still_missing_size = len([t for t in event_tickers if t not in set(firm_size["ticker"])])
    still_missing_sector = len([t for t in event_tickers if t not in set(sector_reference["ticker"])])
    print(f"After gap-fill: firm size missing for {still_missing_size}/{len(event_tickers)}, "
          f"sector missing for {still_missing_sector}/{len(event_tickers)} "
          f"(usually delisted/foreign-private-issuer tickers yfinance has no record for).")

    # Cache the gap-filled rows back to data/raw so re-running this notebook
    # doesn't re-fetch the same tickers every time.
    firm_size.to_csv(FIRM_SIZE_FILE, index=False)
    sector_reference.to_csv(SECTOR_REFERENCE_FILE, index=False)
else:
    print("\nAll event tickers already covered by the index-based reference tables - no gap-fill needed.")
    if "firm_size_source" not in firm_size.columns:
        firm_size["firm_size_source"] = "Index reference (Step 2)"
    if "sector_source" not in sector_reference.columns:
        sector_reference["sector_source"] = sector_reference.get("sector_source", "Index reference (Step 2)")

rows = []
for f in files:
    basename = os.path.basename(f).replace(".csv", "")
    ticker, event_id = basename.split("_", 1)

    df = pd.read_csv(f, index_col=0, parse_dates=True)
    alpha, beta = estimate_market_model(df)
    if alpha is None:
        continue

    car_short = compute_car(df, alpha, beta, SHORT_WINDOW)
    car_long = compute_car(df, alpha, beta, LONG_WINDOW)

    meta = screening[screening["accession_no"] == event_id]
    size_row = firm_size[firm_size["ticker"] == ticker]
    sector_row = sector_reference[sector_reference["ticker"] == ticker]

    rows.append({
        "event_id": event_id,
        "ticker": ticker,
        "event_date": meta["file_date"].iloc[0] if not meta.empty else None,
        "announcement_type": meta["announcement_type"].iloc[0] if not meta.empty else None,
        "firm_size_log": size_row["market_cap_log"].iloc[0] if not size_row.empty else None,
        "firm_size_source": size_row["firm_size_source"].iloc[0] if not size_row.empty else None,
        "sector": sector_row["sector"].iloc[0] if not sector_row.empty else None,
        "sector_source": sector_row["sector_source"].iloc[0] if not sector_row.empty else None,
        "alpha": alpha,
        "beta": beta,
        "CAR_short": car_short,
        "CAR_long": car_long,
    })

analysis_df = pd.DataFrame(rows)

analysis_df["sector_binary"] = analysis_df["sector"].apply(classify_sector_binary)

analysis_df = analysis_df.dropna(subset=["CAR_short"])
analysis_df.to_csv(OUTPUT_FILE, index=False)

n_size = analysis_df["firm_size_log"].notna().sum()
n_sector = analysis_df["sector"].notna().sum()
print(f"\nFinal analysis dataset: {len(analysis_df)} events -> {OUTPUT_FILE}")
print(f"Firm size coverage: {n_size}/{len(analysis_df)} ({n_size/len(analysis_df):.1%})")
print(f"Sector coverage: {n_sector}/{len(analysis_df)} ({n_sector/len(analysis_df):.1%})")
print(f"CAAR (mean CAR_short across sample): {analysis_df['CAR_short'].mean():.4%}")
analysis_df.head()

Processing 447 event return files ...

236/251 event tickers not covered by the index-based reference tables - gap-filling directly via yfinance ...
  gap-fill processed 25/236
  gap-fill processed 50/236
  gap-fill processed 75/236
  gap-fill processed 100/236
  gap-fill processed 125/236
  gap-fill processed 150/236
  gap-fill processed 175/236
  gap-fill processed 200/236
  gap-fill processed 225/236
After gap-fill: firm size missing for 1/251, sector missing for 1/251 (usually delisted/foreign-private-issuer tickers yfinance has no record for).

Final analysis dataset: 445 events -> /content/QM640-WALSH-CAPSTONE/data/processed/analysis_dataset.csv
Firm size coverage: 444/445 (99.8%)
Sector coverage: 444/445 (99.8%)
CAAR (mean CAR_short across sample): -5.2971%


,event_id,ticker,event_date,announcement_type,firm_size_log,firm_size_source,sector,sector_source,alpha,beta,CAR_short,CAR_long,sector_binary
0,0001193125-25-155007:d91487dex991.htm,XRX,2025-07-02,M&A,19.774366,Direct gap-fill (yfinance),Industrials,Direct gap-fill (yfinance),-0.002471,1.363147,-0.042240,-0.240312,Non-Technology
1,0001213900-26-069795:ea029521601ex99-1.htm,MYSE,2026-06-18,partnership,16.210170,Direct gap-fill (yfinance),Technology,Direct gap-fill (yfinance),0.003747,2.119101,0.224378,-0.110368,Technology
2,0001493152-26-008435:ex99-1.htm,KAPA,2026-03-02,M&A,15.695581,Direct gap-fill (yfinance),Healthcare,Direct gap-fill (yfinance),-0.000544,2.442795,0.093387,-0.019670,Non-Technology
3,0001185185-25-000690:mitiex99-1.htm,MITI,2025-06-25,R&D,13.638323,Direct gap-fill (yfinance),Technology,Direct gap-fill (yfinance),0.007699,0.467415,0.079106,-0.746096,Technology
4,0001213900-25-059641:ea024758501ex99-1_richtec...,RR,2025-06-30,M&A,19.576008,Direct gap-fill (yfinance),Industrials,Direct gap-fill (yfinance),0.019502,2.184100,-0.078662,-0.555883,Non-Technology


## Commit and push results back to GitHub

In [7]:
!git -C {BASE_DIR} add "data/processed/analysis_dataset.csv"
!git -C {BASE_DIR} add "data/raw/firm_size.csv"
!git -C {BASE_DIR} add "data/raw/sector_reference.csv"
!git -C {BASE_DIR} commit -m "Step 5: market model, AR, CAR_short, CAR_long, CAAR; gap-fill firm size/sector for non-index-member tickers"
!git -C {BASE_DIR} push


[main 8c7ecb6] Step 5: market model, AR, CAR_short, CAR_long, CAAR; gap-fill firm size/sector for non-index-member tickers
 3 files changed, 1420 insertions(+), 504 deletions(-)
 create mode 100644 data/processed/analysis_dataset.csv
 rewrite data/raw/firm_size.csv (98%)
Enumerating objects: 14, done.
Counting objects: 100% (14/14), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (8/8), 49.10 KiB | 5.46 MiB/s, done.
Total 8 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/Shanmuganathan75/QM640-WALSH-CAPSTONE.git
   7df01ce..8c7ecb6  main -> main


## Sanity check against your N=159 target

In [8]:
print("Final N:", len(analysis_df))
print("Target minimum (RQ2 bottleneck): 159")
print("Practical collection goal: 175-190")
if len(analysis_df) < 159:
    print("\nBelow target - consider widening the EDGAR keyword set/date range in "
          "Step 1, or report the shortfall transparently in Limitations. Do not "
          "force events through screening just to hit the number.")
analysis_df["CAR_short"].describe()

Final N: 445
Target minimum (RQ2 bottleneck): 159
Practical collection goal: 175-190


,CAR_short
count,445.000000
mean,-0.052971
std,1.273823
min,-26.314636
25%,-0.081955
50%,-0.019857
75%,0.049380
max,3.644825
